Run version 1.0 - 30-11-2025

In [1]:
import requests
import pandas as pd
import time
from langdetect import detect
from datetime import datetime

Parameters

In [2]:
# search term in list form
with open("keywords.txt", encoding="utf-8") as f:
    keywords = [word.strip() for word in f.read().split(",") if word.strip()]

keywords2 = ["dental"]

# country of the app store to search in, using ISO country codes: http://en.wikipedia.org/wiki/ISO_3166-1_alpha-2
country = "NL"

# type of media to search, software for application
media_type = "software"

# software for iOS app, iPadSoftware for iPad, macSoftware for MacOS
entity = "software"

# maximum amount of result to return, max = 200
limit = "200"

In [6]:
# language used within the app to keep in the data, using ISO country codes in LOWER CASE: http://en.wikipedia.org/wiki/ISO_3166-1_alpha-2
allowed_langs = ["en", "nl"]

# keep paid apps? True or False
keep_paid_apps = True

# use langdetect for advanced language detection based on app description, and then apply filtering? True or False
lang_detect = True

# the primary app category (in the data "genre") to REMOVE in LOWER CASE, can pick between any of the following:
"""
'Medical' 'Education' 'Games' 'Business' 'Productivity' 'Reference'
 'Health & Fitness' 'Utilities' 'Social Networking' 'Graphics & Design'
 'Shopping' 'Photo & Video' 'Travel' 'Lifestyle' 'News' 'Entertainment'
 'Food & Drink' 'Stickers' 'Music' 'Book' 'Navigation' 'Weather' 'Finance'
 'Sports' 'Magazines & Newspapers'
"""

category_removal = ['games', 'social networking', 'graphics & design', 'shopping', 'photo & video', 'travel', 
                    'news', 'entertainment', 'stickers', 'navigation', 'weather', 'finance', 'sports', 'magazines & newspapers', 'business']


# release date of current version between ("YYYY-MM-DD") or now for date of today
start_date = "2017-09-01"
end_date = "now"

Retreive data

In [7]:
# empty list to append results of different keywords to
all_results = []

if len(keywords) >= 20:
    print(f"More than 20 keywords detected ({len(keywords)}), retreiving the data takes longer because of the API limit of 20/min")
    print(f"Estimated query time: {round((len(keywords)*3.2)/60, 1)} minutes")

for term in keywords:
    # url used to fetch data
    url = f"https://itunes.apple.com/search?term={term}&country={country}&media={media_type}&entity={entity}&limit={limit}&lang=en_us"

    # load data as json
    response = requests.get(url)
    data=response.json()

    # check if there are more or less than 200 results per keyword
    result_count = data.get("resultCount", 0)
    results = data.get("results", [])
    print(f"Query '{term}' returned {result_count} results")
    if result_count >= 200:
        print("!WARNING! for {term}: hit the 200 API limit")
    
    # add the results for which searchterm
    for result_term in results:
        result_term["query_term"] = term
    
    # append to the empty list
    all_results.extend(results)

    # wait 0.5 seconds per query to not overload the api. 20 request per minute is the max, so when there are more than
    # 20 keywords, the query will not hit the limit
    if len(keywords) >= 20:
        time.sleep(3.2)
    else:
        time.sleep(0.5)

df = pd.DataFrame(all_results)
print(f"Total of {df.shape[0]} apps found in initial search for {len(keywords)} keywords")


More than 20 keywords detected (55), retreiving the data takes longer because of the API limit of 20/min
Estimated query time: 2.9 minutes
Query 'dental' returned 161 results
Query 'tooth' returned 142 results
Query 'teeth' returned 143 results
Query 'mouth' returned 83 results
Query 'oral' returned 113 results
Query 'gums' returned 149 results
Query 'dental health' returned 103 results
Query 'dental hygiene' returned 72 results
Query 'dental self-care' returned 1 results
Query 'dental cleaning' returned 36 results
Query 'tooth health' returned 118 results
Query 'tooth hygiene' returned 60 results
Query 'tooth self-care' returned 4 results
Query 'tooth cleaning' returned 87 results
Query 'teeth health' returned 122 results
Query 'teeth hygiene' returned 60 results
Query 'teeth self-care' returned 4 results
Query 'teeth cleaning' returned 87 results
Query 'mouth health' returned 42 results
Query 'mouth hygiene' returned 12 results
Query 'mouth self-care' returned 0 results
Query 'mouth 

Filtering

In [8]:
initial_amount = df.shape[0]
print(f"Initial amount in dataset: {initial_amount}")


# remove duplicates
df = df.drop_duplicates(subset="trackId", keep="first")
print(f"Filtered {initial_amount - df.shape[0]} duplicates")



# only select the apps that have EN, NL or nothing as language
# keep in mind the value in the languageCodesISO2A column is a list.
df["languageCodesISO2A"] = df["languageCodesISO2A"].apply(
    lambda langs: [s.lower() for s in langs] if isinstance(langs, list) else langs
)
filter_country = df["languageCodesISO2A"].apply(
    lambda langs: (
        isinstance(langs, list) and
        (any(lang in langs for lang in allowed_langs) or len(langs) == 0)
    )
)
# Count and print how many rows were filtered out
num_filtered_out = (~filter_country).sum()
print(f"Filtered {num_filtered_out} apps based on given app language(apps without {allowed_langs}, unspecified languages kept)")
df = df[filter_country]



# filter based on description language
if lang_detect:
    def detect_language_safe(text):
        try:
            if not isinstance(text, str) or text.strip() == "":
                return "unknown"
            return detect(text)
        except Exception:
            return "unknown"

    # Apply detection to DataFrame
    df["detected_lang"] = df["description"].apply(detect_language_safe)

    # Filter based on allowed languages
    filter_lang = df["detected_lang"].apply(lambda x: x in allowed_langs or x == "unknown")

    num_filtered_out = (~filter_lang).sum()
    print(f"Filtered {num_filtered_out} apps with description not in {allowed_langs} language")

    df = df[filter_lang]
    


# filter on available for iphone
filtered_devices = df["supportedDevices"].apply(
    lambda devices: isinstance(devices, list) and any("iphone" in d.lower() for d in devices)
)
print(f"Filtered {(filtered_devices == False).sum()} apps that are not available on iPhone")
df = df[filtered_devices]

if not keep_paid_apps:
    print(f"Filtered {(df[df["price"] != 0.0]).shape[0]} paid apps")
    df = df[df["price"] == 0.0]


# filter title or description includes any of the keywords
# create regex pattern
pattern = "|".join(keywords)
filtered_keywords = (
    df["trackName"].str.lower().str.contains(pattern, regex=True, na=False) |
    df["description"].str.lower().str.contains(pattern, regex=True, na=False)
)
print(f"Filtered {(~filtered_keywords).sum()} apps that do NOT contain any of the given keywords")
df = df[filtered_keywords]



# filter primary genre name to remove some value
filter_prime_cat = ~df["primaryGenreName"].str.lower().apply(lambda x: x in category_removal)
print(f"Filtered {(~filter_prime_cat).sum()} apps with not allowed primary category")
df = df[filter_prime_cat]


# filter based on release date of latest version
df["currentVersionReleaseDate"] = pd.to_datetime(df["currentVersionReleaseDate"], errors="coerce")

start_date_ = pd.Timestamp(start_date, tz="UTC")
if end_date == "now":
    end_date_ = pd.Timestamp.now(tz="UTC")
else:
    end_date_ = pd.Timestamp(end_date, tz="UTC")
filtered_date = (df["currentVersionReleaseDate"] >= start_date_) & (df["currentVersionReleaseDate"] <= end_date_)
print(f"Filtered {(~filtered_date).sum()} apps updated before {start_date_.date()} and after {end_date_.date()}")
df = df[filtered_date]


print(f"Results after filtering: {df.shape[0]}")



Initial amount in dataset: 3444
Filtered 1448 duplicates
Filtered 55 apps based on given app language(apps without ['en', 'nl'], unspecified languages kept)
Filtered 136 apps with description not in ['en', 'nl'] language
Filtered 0 apps that are not available on iPhone
Filtered 707 apps that do NOT contain any of the given keywords
Filtered 396 apps with not allowed primary category
Filtered 11 apps updated before 2017-09-01 and after 2025-11-30
Results after filtering: 691


C:\Users\Niek\AppData\Local\Temp\ipykernel_4944\778657989.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["currentVersionReleaseDate"] = pd.to_datetime(df["currentVersionReleaseDate"], errors="coerce")


In [11]:
df["currentVersionReleaseDate"] = df["currentVersionReleaseDate"].dt.tz_localize(None)

import re

def clean_excel_chars(x):
    if isinstance(x, str):
        # remove ASCII forbidden chars
        return re.sub(r"[\x00-\x08\x0B-\x1F\x7F]", "", x)
    return x

df = df.applymap(clean_excel_chars)

df.to_excel("ios_data_v1_0.xlsx", index=False)

C:\Users\Niek\AppData\Local\Temp\ipykernel_4944\2583070805.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_excel_chars)
